# 09 — Validation
Runs every generated code sample through `aro check` (and optionally `aro compile`).
Only samples whose ARO code passes are kept in the validated set.
Debugging samples (task_type=debugging) are validated against their *fix* block,
and the buggy block is verified to *fail* check.

**Input:**  `../data/03_raw_generated/samples.jsonl`  (from notebook 08)
**Output:** `../data/04_validated/valid_samples.jsonl`
            `../data/04_validated/all_samples.jsonl`   (includes invalid, for analysis)

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json
import re
import subprocess
import tempfile
from pathlib import Path
from collections import Counter, defaultdict

DATA_OUT = DATA_ROOT / '04_validated'
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Load samples from notebook 08 output (03_raw_generated/samples.jsonl)
RAW_DATA_DIR = DATA_ROOT / '03_raw_generated'
samples_file = RAW_DATA_DIR / 'samples.jsonl'

if not samples_file.exists():
    raise FileNotFoundError(
        f'samples.jsonl not found at {samples_file}\n'
        f'Run notebook 08 (synthetic data generation) first.'
    )

samples = []
with open(samples_file) as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f'Loaded {len(samples)} samples from {samples_file}')

## Check aro availability

In [ ]:
def aro_available():
    try:
        r = subprocess.run(['aro', '--version'], capture_output=True, timeout=5)
        return r.returncode == 0
    except FileNotFoundError:
        return False

ARO_OK = aro_available()
print(f'aro CLI available: {ARO_OK}')
if not ARO_OK:
    print('WARNING: aro not in PATH. Code samples will be marked valid=None (skipped).')

## Validation helpers

In [ ]:
def extract_aro_blocks(text):
    """Extract all ```aro ... ``` code blocks from a string."""
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]

def extract_yaml_blocks(text):
    return [b.strip() for b in re.findall(r'```yaml\n(.*?)```', text, re.DOTALL) if b.strip()]

def aro_check(code, extra_files=None):
    """
    Write code to a temp directory and run `aro check`.
    extra_files: dict of {filename: content} to write alongside main.aro
    Returns (passed: bool | None, error: str)
      None means aro is not available.
    """
    if not ARO_OK:
        return None, 'aro_not_available'
    with tempfile.TemporaryDirectory() as tmp:
        tmpdir = Path(tmp)
        (tmpdir / 'main.aro').write_text(code)
        if extra_files:
            for name, content in extra_files.items():
                (tmpdir / name).write_text(content)
        try:
            r = subprocess.run(
                ['aro', 'check', str(tmpdir)],
                capture_output=True, text=True, timeout=10,
            )
            if r.returncode == 0:
                return True, ''
            return False, (r.stderr or r.stdout).strip()[:300]
        except subprocess.TimeoutExpired:
            return False, 'timeout'
        except Exception as e:
            return False, str(e)

## Per-task validation logic

In [ ]:
def validate(sample):
    task = sample.get('task_type', '')
    output = sample.get('output', '')

    # FIM: no code to validate; accept as-is
    if task == 'fim':
        return True, '', {}

    # Explanation / architecture / syntax_qa: no executable code
    if task in ('code_explanation', 'syntax_qa', 'architecture'):
        return True, '', {}

    # --- Tasks that produce ARO code ---
    aro_blocks = extract_aro_blocks(output)
    yaml_blocks = extract_yaml_blocks(output)
    extra = {'openapi.yaml': yaml_blocks[0]} if yaml_blocks else {}

    if not aro_blocks:
        return False, 'no_aro_code_found', {}

    combined_aro = '\n\n'.join(aro_blocks)

    if task == 'debugging':
        # Validate the FIX block specifically
        fix_match = re.search(r'## Fix\s*```aro\n(.*?)```', output, re.DOTALL)
        buggy_match = re.search(r'## Buggy Code\s*```aro\n(.*?)```', output, re.DOTALL)

        fix_code   = fix_match.group(1).strip() if fix_match else combined_aro
        buggy_code = buggy_match.group(1).strip() if buggy_match else ''

        passed, error = aro_check(fix_code, extra)
        meta = {}
        if buggy_code:
            buggy_passed, _ = aro_check(buggy_code, extra)
            meta['buggy_correctly_fails'] = (buggy_passed is False)
        return passed, error, meta

    # code_generation, translation
    passed, error = aro_check(combined_aro, extra)
    return passed, error, {}

## Run validation

In [ ]:
validated = []
stats = Counter()

for i, sample in enumerate(samples):
    passed, error, meta = validate(sample)

    s = dict(sample)
    s['valid'] = passed
    s['validation_error'] = error
    s.update(meta)
    validated.append(s)

    if passed is True:
        stats['valid'] += 1
    elif passed is None:
        stats['skipped'] += 1
    elif error == 'no_aro_code_found':
        stats['no_code'] += 1
    else:
        stats['invalid'] += 1

    if (i + 1) % 100 == 0:
        pct = 100 * stats['valid'] // max(1, i + 1 - stats['skipped'])
        print(f'  {i+1}/{len(samples)}  valid={stats["valid"]}  invalid={stats["invalid"]}  ({pct}% pass rate)')

print(f'\nDone.')

## Save results

In [ ]:
# All samples (for error analysis)
with open(DATA_OUT / 'all_samples.jsonl', 'w') as f:
    for s in validated:
        f.write(json.dumps(s) + '\n')

# Valid only
valid_samples = [s for s in validated if s['valid'] is not False]
with open(DATA_OUT / 'valid_samples.jsonl', 'w') as f:
    for s in valid_samples:
        f.write(json.dumps(s) + '\n')

total = len(validated)
print(f'Results saved to {DATA_OUT}')
print(f'  Total:   {total}')
print(f'  Valid:   {stats["valid"]:4d}  ({100*stats["valid"]//total}%)')
print(f'  Invalid: {stats["invalid"]:4d}')
print(f'  No code: {stats["no_code"]:4d}')
print(f'  Skipped: {stats["skipped"]:4d}  (aro not available)')

## Error analysis

In [ ]:
error_counts = Counter()
for s in validated:
    if not s['valid']:
        err = s.get('validation_error', 'unknown')
        # Normalise common error patterns
        for pattern, label in [
            (r'undefined variable', 'undefined_variable'),
            (r'expected', 'parse_error_expected'),
            (r'Application-Start', 'missing_application_start'),
            (r'operationId', 'operationId_mismatch'),
        ]:
            if re.search(pattern, err, re.IGNORECASE):
                err = label
                break
        error_counts[err[:80]] += 1

print('Top validation errors:')
for err, n in error_counts.most_common(15):
    print(f'  {n:4d}x  {err}')

# Per-task pass rates
task_stats = defaultdict(lambda: {'total': 0, 'valid': 0})
for s in validated:
    t = s.get('task_type', 'unknown')
    task_stats[t]['total'] += 1
    if s['valid'] is not False:
        task_stats[t]['valid'] += 1

print('\nPass rate by task type:')
for task, ts in sorted(task_stats.items()):
    rate = 100 * ts['valid'] // max(1, ts['total'])
    print(f'  {task:25s}: {ts["valid"]}/{ts["total"]}  ({rate}%)')